In [ ]:
# US Superstore - Analyse exploratoire, diagnostics et tableau de bord interactif
# (Coller l’ensemble de ce bloc dans un notebook. Les cellules peuvent être scindées si besoin.)

# =========================
# 1) Portée et préparation
# =========================

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset (placer superstore_dataset.csv dans le même dossier que le notebook)
df = pd.read_csv('superstore_dataset.csv')

# Exploration de base
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nInfo:")
df.info()
print("\nDescribe (numériques):")
display(df.describe())
print("\nValeurs manquantes par colonne:")
display(df.isnull().sum())

# Nettoyage et prétraitement
print("\nDoublons:", df.duplicated().sum())
df = df.drop_duplicates()

# Gestion simple des valeurs manquantes (adapter selon votre dataset)
# Exemple: Postal Code manquant
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

# Conversion des dates
date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("\nTypes après conversion des dates:")
print(df[ [c for c in date_columns if c in df.columns] ].dtypes)

# =========================
# 1-b) Feature engineering
# =========================

# Hypothèse: le dataset contient Sales et Profit, Category, State, Product Name, Discount
# Créer Profit Margin, Year, Month, Period mensuel
if {'Sales','Profit'}.issubset(df.columns):
    df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
if 'Order Date' in df.columns:
    df['Order Year'] = df['Order Date'].dt.year
    df['Order Month'] = df['Order Date'].dt.month
    df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print("\nNouvelles features créées (aperçu):")
cols_show = [c for c in ['Sales','Profit','Profit Margin','Order Year','Order Month'] if c in df.columns]
display(df[cols_show].head())

# ================================
# 2) Analyse exploratoire avancée
# ================================

# 2-a) Série temporelle: ventes mensuelles (toutes catégories et par catégorie)
if {'Order Month-Year','Sales'}.issubset(df.columns):
    monthly_sales = df.groupby(['Order Month-Year'] + (['Category'] if 'Category' in df.columns else []))['Sales'] \
                      .sum().reset_index()
    monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

    def plot_monthly_sales(category='All'):
        plt.figure(figsize=(12, 6))
        if category == 'All' or 'Category' not in df.columns:
            total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
            plt.plot(total_monthly.index.to_timestamp(), total_monthly.values, marker='o', linewidth=2, markersize=4)
            plt.title('Monthly Sales Trend - All Categories')
        else:
            category_data = monthly_sales[monthly_sales['Category'] == category]
            plt.plot(category_data['Date'], category_data['Sales'], marker='o', linewidth=2, markersize=4)
            plt.title(f'Monthly Sales Trend - {category}')
        plt.xlabel('Date')
        plt.ylabel('Sales ($)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    categories = ['All'] + (list(df['Category'].dropna().unique()) if 'Category' in df.columns else [])
    category_dropdown = Dropdown(options=categories, value='All', description='Category:')
    interact(plot_monthly_sales, category=category_dropdown)

# 2-b) Performance géographique: ventes par Etat (Top N)
if {'State','Sales'}.issubset(df.columns):
    state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

    def plot_top_states(top_n=10):
        plt.figure(figsize=(12, max(6, top_n * 0.4)))
        top_states = state_sales.tail(top_n)
        bars = plt.barh(range(len(top_states)), top_states.values)
        plt.yticks(range(len(top_states)), top_states.index)
        plt.xlabel('Total Sales ($)')
        plt.ylabel('State')
        plt.title(f'Top {top_n} States by Sales Performance')

        for i, (state, value) in enumerate(top_states.items()):
            plt.text(value + max(top_states.values()) * 0.01, i, f'${value:,.0f}', va='center', fontsize=9)

        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Total states analyzed: {len(state_sales)}")
        print(f"Top {top_n} states represent: ${top_states.sum():,.0f} in sales")

    top_n_slider = IntSlider(min=5, max=25, value=10, description='Top N:')
    interact(plot_top_states, top_n=top_n_slider)
else:
    state_sales = pd.Series(dtype=float)  # fallback vide si pas de colonne State

# ============================
# 3) Communication d’insights
# ============================

# 3-a) Top 10 produits les plus rentables (Seaborn)
if {'Product Name','Profit'}.issubset(df.columns):
    product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x=product_profit.values, y=product_profit.index, orient='h')
    plt.title('Top 10 Most Profitable Products')
    plt.xlabel('Total Profit ($)')
    plt.ylabel('Product Name')

    for i, (product, profit) in enumerate(product_profit.items()):
        ax.text(profit + max(product_profit.values()) * 0.01, i, f'${profit:,.0f}', va='center', fontsize=9)

    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Key Insights:")
    print(f"- Most profitable product: {product_profit.index[0]} (${product_profit.iloc[0]:,.0f})")
    print(f"- Top 10 products total profit: ${product_profit.sum():,.0f}")
    print(f"- Average profit among top 10: ${product_profit.mean():,.0f}")

# 3-b) Remise vs Profit (scatter + tendance)
if {'Discount','Profit'}.issubset(df.columns):
    plt.figure(figsize=(14, 8))
    if 'Category' in df.columns:
        sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=50)
    else:
        sns.scatterplot(data=df, x='Discount', y='Profit', alpha=0.6, s=50)

    sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red',
                line_kws={'linewidth': 2, 'linestyle': '--'})

    plt.title('Discount vs Profit')
    plt.xlabel('Discount')
    plt.ylabel('Profit ($)')
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    plt.grid(True, alpha=0.3)
    if 'Category' in df.columns:
        plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    high_discount = df[df['Discount'] > 0.2]
    print("Discount Analysis Insights:")
    print(f"- Transactions with >20% discount: {len(high_discount):,}")
    print(f"- Average profit for >20% discount: ${high_discount['Profit'].mean():.2f}")
    print(f"- Share of losses when discount >20%: {(high_discount['Profit'] < 0).mean()*100:.1f}%")

    if 'Category' in df.columns:
        print("\nCategory-specific impact (>20% discount):")
        for category in df['Category'].dropna().unique():
            cat_data = df[(df['Category'] == category) & (df['Discount'] > 0.2)]
            if len(cat_data) > 0:
                print(f"- {category}: avg profit = ${cat_data['Profit'].mean():.2f}")

# ================================
# 4) Méthodologie et comparaison
# ================================

print("\n=== LIBRARY COMPARISON ANALYSIS ===\n")

print("MATPLOTLIB STRENGTHS (from our analysis):")
print("- Fine-grained control over interactive widgets")
print("- Custom annotations and text positioning")
print("- Precise subplot layouts and figure sizing")
print("- Integration with ipywidgets for dynamic updates\n")

print("SEABORN STRENGTHS (from our analysis):")
print("- Built-in statistical visualizations (regplot)")
print("- Automatic color palettes and legends")
print("- Clean default styling")
print("- Easy categorical plots\n")

# Micro comparaison de temps d’exécution (indicative, non scientifique)
import time
if 'Order Year' in df.columns and 'Sales' in df.columns:
    start = time.time()
    plt.figure(figsize=(8, 6))
    plt.plot(df.groupby('Order Year')['Sales'].sum())
    plt.close()
    matplotlib_time = time.time() - start

    start = time.time()
    plt.figure(figsize=(8, 6))
    sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(),
                 x='Order Year', y='Sales')
    plt.close()
    seaborn_time = time.time() - start

    print("SPEED COMPARISON:")
    print(f"- Matplotlib basic plot: {matplotlib_time:.4f} s")
    print(f"- Seaborn equivalent:    {seaborn_time:.4f} s")
else:
    print("SPEED COMPARISON skipped (Order Year or Sales missing).")

# ============================
# 5) Executive summary (auto)
# ============================

print("\n=== EXECUTIVE SUMMARY - KEY FINDINGS ===\n")
if {'Sales','Profit'}.issubset(df.columns):
    total_sales = df['Sales'].sum()
    total_profit = df['Profit'].sum()
    profit_margin = (total_profit / total_sales) * 100 if total_sales != 0 else np.nan

    print("BUSINESS PERFORMANCE:")
    print(f"- Total Revenue: ${total_sales:,.0f}")
    print(f"- Total Profit: ${total_profit:,.0f}")
    print(f"- Overall Profit Margin: {profit_margin:.1f}%\n")
else:
    print("BUSINESS PERFORMANCE: metrics unavailable (missing Sales/Profit).\n")

if len(state_sales) > 0 and 'Sales' in df.columns:
    top_state = state_sales.index[-1]
    top_state_sales = state_sales.iloc[-1]
    print("GEOGRAPHIC PERFORMANCE:")
    print(f"- Top performing state: {top_state} (${top_state_sales:,.0f})")
    print(f"- Top 5 states = {(state_sales.tail(5).sum()/df['Sales'].sum())*100:.1f}% of sales\n")

if {'Category','Sales'}.issubset(df.columns):
    top_category = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).index[0]
    print("PRODUCT PERFORMANCE:")
    print(f"- Leading category: {top_category}")
    if 'Product Name' in df.columns and 'Profit' in df.columns:
        if 'product_profit' not in locals():
            product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)
        print(f"- Most profitable product: {product_profit.index[0]}\n")
    else:
        print("- Most profitable product: unavailable (missing Profit/Product Name)\n")

if {'Discount','Profit'}.issubset(df.columns):
    high_discount_loss_rate = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100
    print("DISCOUNT STRATEGY INSIGHTS:")
    print(f"- High discount loss rate (>20%): {high_discount_loss_rate:.1f}%")
    print("- Recommended max standard discount: around 20%\n")

# ============================
# Options avancées (facultatif)
# ============================

# Tableau de bord multi-graphes
def create_dashboard():
    if not {'Order Month-Year','Sales'}.issubset(df.columns):
        print("Dashboard indisponible: Order Month-Year ou Sales manquants.")
        return

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    # 1) Tendance des ventes mensuelles
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values, marker='o')
    ax1.set_title('Monthly Sales Trend')
    ax1.tick_params(axis='x', rotation=45)

    # 2) Ventes par catégorie
    if {'Category','Sales'}.issubset(df.columns):
        category_sales = df.groupby('Category')['Sales'].sum()
        ax2.bar(category_sales.index, category_sales.values)
        ax2.set_title('Sales by Category')
        ax2.tick_params(axis='x', rotation=30)
    else:
        ax2.set_title('Sales by Category (indisponible)')

    # 3) Top 10 Etats
    if len(state_sales) > 0:
        top_10_states = state_sales.tail(10)
        ax3.barh(range(len(top_10_states)), top_10_states.values)
        ax3.set_yticks(range(len(top_10_states)))
        ax3.set_yticklabels(top_10_states.index)
        ax3.set_title('Top 10 States by Sales')
    else:
        ax3.set_title('Top 10 States by Sales (indisponible)')

    # 4) Discount vs Profit
    if {'Discount','Profit'}.issubset(df.columns):
        if 'Category' in df.columns:
            for category in df['Category'].dropna().unique():
                cat_data = df[df['Category'] == category]
                ax4.scatter(cat_data['Discount'], cat_data['Profit'], label=category, alpha=0.6)
            ax4.legend()
        else:
            ax4.scatter(df['Discount'], df['Profit'], alpha=0.6)
        ax4.set_xlabel('Discount')
        ax4.set_ylabel('Profit')
        ax4.set_title('Discount vs Profit')
        ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    else:
        ax4.set_title('Discount vs Profit (indisponible)')

    plt.tight_layout()
    plt.show()

# create_dashboard()  # décommenter pour afficher

# Annotation des outliers sur Discount vs Profit
if {'Discount','Profit'}.issubset(df.columns):
    plt.figure(figsize=(12, 8))
    if 'Category' in df.columns:
        sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6)
    else:
        sns.scatterplot(data=df, x='Discount', y='Profit', alpha=0.6)

    top_3 = df.nlargest(3, 'Profit')
    worst_3 = df.nsmallest(3, 'Profit')

    for _, row in top_3.iterrows():
        plt.annotate(f'Best ${row["Profit"]:.0f}', xy=(row['Discount'], row['Profit']),
                     xytext=(10, 10), textcoords='offset points',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.8),
                     arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

    for _, row in worst_3.iterrows():
        plt.annotate(f'Worst ${row["Profit"]:.0f}', xy=(row['Discount'], row['Profit']),
                     xytext=(10, -20), textcoords='offset points',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='salmon', alpha=0.8),
                     arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

    plt.title('Discount vs Profit with Outlier Annotations')
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# Version Plotly (facultatif, nécessite plotly)
# import plotly.express as px
# fig = px.scatter(df, x='Discount', y='Profit', color='Category' if 'Category' in df.columns else None,
#                  hover_data=['Product Name','Sales'] if {'Product Name','Sales'}.issubset(df.columns) else None,
#                  title='Interactive Discount vs Profit (Plotly)')
# fig.show()
